# Credit Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Package leverage, coverage, covenant, and per-instrument credit metrics for credit-committee review.

**Prerequisites:** `04_statement_modeling/models/credit_analysis.ipynb` and `04_statement_modeling/models/covenant_monitoring.ipynb`.

**What you'll learn:**

- Prepare credit metrics and covenant status inputs.
- Render `reporting.credit_tearsheet` for a borrower or issuer.
- Connect model analytics to a concise credit narrative.

Build a borrower's quarterly model, compute a structured `credit_assessment`
(leverage, interest coverage, free cash flow, plus a per-period series), and
render a `credit_tearsheet`. Per-instrument coverage and covenant headroom are
deal terms supplied by the caller (or sourced from your credit engine).


In [1]:
import datetime as dt
import json

import sys
sys.path.insert(0, "..")

from _shared import demo_pl_model
from finstack_quant import reporting
from finstack_quant.covenants import evaluate_engine
from finstack_quant.statements import Evaluator
from finstack_quant.statements_analytics import credit_assessment

# The example suite's shared quarterly P&L, extended with the balance-sheet and
# cash lines `credit_assessment` needs. Margin lines are dropped: a credit sheet
# reads leverage and coverage, not gross/EBITDA margin.
borrower = demo_pl_model(
    "borrower",
    margins=False,
    values={
        "interest_expense": [4.0] * 8,
        "total_debt": [300, 300, 295, 295, 290, 290, 285, 285],
        "free_cash_flow": [8, 9, 10, 11, 12, 13, 14, 15],
    },
)
result = Evaluator().evaluate(borrower)

assessment = credit_assessment(result.to_json(), "2025Q4")
print("Leverage:", round(assessment["leverage_ratio"], 2), " Coverage:", round(assessment["interest_coverage"], 2))


Leverage: 2.01  Coverage: 8.88


## Credit profile

Leverage/coverage trend and free cash flow come from the structured
`credit_assessment`. Per-instrument coverage (DSCR / interest cover / LTV) is
passed in as deal terms. (LTV is expressed in percent units, e.g. `55.0`
renders as `55.0%`.)

Covenant compliance is **not** hand-authored: the deal's covenant thresholds and
the reported metric values are the only fixture data, and
`covenants.evaluate_engine` decides pass/fail and computes headroom. Headroom is
the engine's convention — the signed distance from the threshold **normalised by
the threshold**, so `3.1x` against a `4.00x` maximum is `+0.225` (22.5% of the
limit still unused), not `+0.9`.

In [2]:
# Fixture data: covenant type, threshold, and the metric value reported for the
# test date. Everything else on the covenant table comes from the engine.
COVENANT_TERMS = [
    # label, covenant type, threshold, metric id, reported metric value
    ("Max Net Leverage", "MaxNetDebtToEBITDA", 4.0, "net_leverage", 3.1),
    ("Min Interest Coverage", "MinInterestCoverage", 2.5, "interest_coverage", 3.2),
    ("Min DSCR", "MinDSCR", 1.5, "dscr", 1.4),
]


def covenant_spec(label, covenant_type, threshold, metric_id):
    """Wire one quarterly maintenance covenant for the covenants engine."""
    return {
        "covenant": {
            "covenant_type": {covenant_type: {"threshold": threshold}},
            "test_frequency": {"count": 3, "unit": "months"},
            "cure_period_days": 30,
            "consequences": [],
            "is_active": True,
            "scope": "Maintenance",
            "springing_condition": None,
            # The label becomes the covenant's identity key, so reports come
            # back keyed by the name we want on the tear sheet.
            "label": label,
        },
        "metric_id": metric_id,
    }


engine = {
    "specs": [covenant_spec(label, ctype, thr, mid) for label, ctype, thr, mid, _ in COVENANT_TERMS],
    "breach_history": [],
    "windows": [],
    "waivers": [],
}
metrics = {mid: value for *_, mid, value in COVENANT_TERMS}
reports = json.loads(evaluate_engine(json.dumps(engine), json.dumps(metrics), "2025-12-31"))

covenants = [
    {
        "covenant": label,
        "threshold": reports[label]["threshold"],
        "current": reports[label]["actual_value"],
        "headroom": reports[label]["headroom"],
        "status": "Pass" if reports[label]["passed"] else "Breach",
    }
    for label, *_ in COVENANT_TERMS
]
for row in covenants:
    print(f"{row["covenant"]:<24} {row["status"]:<7} headroom {row["headroom"]:+.3f}")

coverage = [
    {"instrument": "Term Loan B", "dscr": 1.8, "interest_coverage": 3.2, "ltv": 55.0},
    {"instrument": "Senior Notes", "dscr": 2.1, "interest_coverage": 3.8, "ltv": 42.0},
]
reporting.credit_tearsheet(
    assessment,
    results=result,
    coverage=coverage,
    covenants=covenants,
    title="Acme Corp — Credit",
    generated=dt.date(2026, 6, 22),
)

Max Net Leverage         Pass    headroom +0.225
Min Interest Coverage    Pass    headroom +0.280
Min DSCR                 Breach  headroom -0.067


TearSheet(theme=Theme(name='institutional', font_head='Georgia,"Times New Roman",serif', font_num='"Iowan Old Style",Georgia,serif', font_sans="-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif", ink='#10243f', muted='#7a8190', pos='#1b6b4f', neg='#9b2335', accent='#7c2230', canvas='#fbfaf6', rule='#10243f', faint='#e4e1d8', grid='#cfc8b8'), title='Acme Corp — Credit', sections=[Section(title='Leverage & Coverage', body='<div class="grid2"><div><svg viewBox="0 0 620 190" xmlns="http://www.w3.org/2000/svg"><line x1="48" y1="166.0" x2="606" y2="166.0" stroke="#e4e1d8"/><text x="42" y="169.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2.0</text><line x1="48" y1="135.2" x2="606" y2="135.2" stroke="#e4e1d8"/><text x="42" y="138.2" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2.1</text><line x1="48" y1="104.4" x2="606" y2="104.4" stroke="#e4e1d8"/><text x="42" y="107.4" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2.2</text><line x1="48" y1="73.6" x2="606" y2="73.6" stroke="#e4e1d8"/><text x="42" y="76.6" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2.3</text><line x1="48" y1="42.8" x2="606" y2="42.8" stroke="#e4e1d8"/><text x="42" y="45.8" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2.4</text><line x1="48" y1="12.0" x2="606" y2="12.0" stroke="#e4e1d8"/><text x="42" y="15.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2.5</text><line x1="48.0" y1="166" x2="48.0" y2="170" stroke="#7a8190"/><text x="48.0" y="181" text-anchor="middle" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2024</text><line x1="366.9" y1="166" x2="366.9" y2="170" stroke="#7a8190"/><text x="366.9" y="181" text-anchor="middle" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">2025</text><polyline points="287.1,12.0 366.9,61.7 446.6,94.9 526.3,136.6 606.0,163.8" fill="none" stroke="#10243f" stroke-width="1.6" stroke-linejoin="round"/><rect class="fq-hb" x="217.4" y="12" width="139.5" height="154" data-cx="287.1" data-cy="12.0" data-label="2024Q4" data-val="2.50"><title>2024Q4 · 2.50</title></rect><rect class="fq-hb" x="297.1" y="12" width="139.5" height="154" data-cx="366.9" data-cy="61.7" data-label="2025Q1" data-val="2.34"><title>2025Q1 · 2.34</title></rect><rect class="fq-hb" x="376.8" y="12" width="139.5" height="154" data-cx="446.6" data-cy="94.9" data-label="2025Q2" data-val="2.23"><title>2025Q2 · 2.23</title></rect><rect class="fq-hb" x="456.5" y="12" width="139.5" height="154" data-cx="526.3" data-cy="136.6" data-label="2025Q3" data-val="2.10"><title>2025Q3 · 2.10</title></rect><rect class="fq-hb" x="536.2" y="12" width="139.5" height="154" data-cx="606.0" data-cy="163.8" data-label="2025Q4" data-val="2.01"><title>2025Q4 · 2.01</title></rect><line class="fq-cross" x1="0" x2="0" y1="12" y2="166" style="visibility:hidden" pointer-events="none"/><circle class="fq-mk" r="3.5" cx="0" cy="0" style="visibility:hidden" pointer-events="none"/></svg></div><div><svg viewBox="0 0 620 190" xmlns="http://www.w3.org/2000/svg"><line x1="48" y1="166.0" x2="606" y2="166.0" stroke="#e4e1d8"/><text x="42" y="169.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">7.0</text><line x1="48" y1="127.5" x2="606" y2="127.5" stroke="#e4e1d8"/><text x="42" y="130.5" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">7.5</text><line x1="48" y1="89.0" x2="606" y2="89.0" stroke="#e4e1d8"/><text x="42" y="92.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">8.0</text><line x1="48" y1="50.5" 

## Saving a standalone HTML file

```python
ts = reporting.credit_tearsheet(assessment, results=result, generated=dt.date(2026, 6, 22))
ts.save("credit_tearsheet.html")
```

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
